# 07 — Upload Embeddings to Qdrant Cloud

**Fruvia AI** — Qdrant Cloud upload notebook.

This notebook runs on **Google Colab** and uploads DINOv2 embeddings
(generated by notebook 06) to Qdrant Cloud.

### What this notebook does

1. Read Qdrant credentials from **Colab Secrets** (never hardcoded)
2. Load embedding shards from Google Drive
3. Create or verify the Qdrant collection
4. Upload vectors in batches with retry logic
5. Support resume (skip already-uploaded points)
6. Verify upload: count points, run sample query

### Prerequisites

- Embedding shards from notebook 06 saved on Google Drive
- Qdrant Cloud credentials stored in Colab Secrets:
  - `QDRANT_URL` — e.g. `https://your-cluster.qdrant.io:6333`
  - `QDRANT_API_KEY` — your API key
- No GPU required — CPU runtime is sufficient

### Security

⚠️ **Never paste API keys into notebook cells.** Always use Colab Secrets.

## 1. Setup & Dependencies

In [ ]:
!pip install -q qdrant-client numpy tqdm

In [ ]:
import json
import time
import uuid
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

## 2. Configuration

In [ ]:
# ── Qdrant Collection ──
COLLECTION_NAME = "fruvia_images_dinov2_base_v1"
VECTOR_SIZE = 768

# ── Paths (Google Drive) ──
EMBEDDINGS_DIR = Path("/content/drive/MyDrive/fruvia-ai/embeddings")

# ── Upload ──
UPLOAD_BATCH_SIZE = 100  # Points per upsert call
MAX_RETRIES = 3  # Retries per failed batch
RETRY_DELAY = 5  # Seconds between retries

# ── Flags ──
RESET_COLLECTION = False  # Set True to delete and recreate collection

## 3. Connect to Qdrant Cloud

In [ ]:
from google.colab import drive, userdata
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

# Mount Google Drive for embedding shards
drive.mount("/content/drive")

# Read credentials from Colab Secrets (NEVER hardcode)
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

assert QDRANT_URL, "QDRANT_URL not found in Colab Secrets"
assert QDRANT_API_KEY, "QDRANT_API_KEY not found in Colab Secrets"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

# Verify connection
collections = client.get_collections().collections
print("Connected to Qdrant Cloud")
print(f"Existing collections: {[c.name for c in collections]}")

## 4. Create or Verify Collection

In [ ]:
existing_names = [c.name for c in client.get_collections().collections]

if RESET_COLLECTION and COLLECTION_NAME in existing_names:
    print(f"⚠️ RESETTING collection '{COLLECTION_NAME}'...")
    client.delete_collection(COLLECTION_NAME)
    existing_names.remove(COLLECTION_NAME)
    print("  Collection deleted.")

if COLLECTION_NAME not in existing_names:
    print(f"Creating collection '{COLLECTION_NAME}'...")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE,
        ),
    )
    print(f"  Collection created: {VECTOR_SIZE}-dim, cosine distance")
else:
    info = client.get_collection(COLLECTION_NAME)
    print(f"Collection '{COLLECTION_NAME}' already exists")
    print(f"  Points: {info.points_count}")
    print(f"  Vectors: {info.vectors_count}")

## 5. Load Embedding Shards

In [ ]:
shard_files = sorted(EMBEDDINGS_DIR.glob("shard_*.npz"))
meta_files = sorted(EMBEDDINGS_DIR.glob("shard_*_meta.json"))

print(f"Found {len(shard_files)} embedding shards")
print(f"Found {len(meta_files)} metadata files")

assert len(shard_files) == len(meta_files), (
    f"Mismatch: {len(shard_files)} shards vs {len(meta_files)} metadata files"
)

# Preview
total_vectors = 0
for sf in shard_files:
    data = np.load(sf)
    n = data["embeddings"].shape[0]
    total_vectors += n
    print(f"  {sf.name}: {n} vectors")
print(f"Total vectors to upload: {total_vectors}")

## 6. Check Already Uploaded (Resume Support)

In [ ]:
# Get current point count for resume
collection_info = client.get_collection(COLLECTION_NAME)
existing_count = collection_info.points_count
print(f"Points already in collection: {existing_count}")

if existing_count > 0 and existing_count >= total_vectors:
    print("\n✅ All vectors appear to be already uploaded!")
    print("Set RESET_COLLECTION=True and re-run to re-upload.")

## 7. Upload Vectors in Batches

In [ ]:
UUID_NAMESPACE = uuid.UUID("a3f1b2c4-d5e6-4f7a-8b9c-0d1e2f3a4b5c")


def make_point_id(image_id: str) -> str:
    """Convert image_id to a stable UUID for Qdrant point ID."""
    # image_id is already a UUID5 string from the manifest
    return image_id


def upload_batch_with_retry(points: list[PointStruct], batch_num: int) -> bool:
    """Upload a batch of points with retry logic."""
    for attempt in range(MAX_RETRIES):
        try:
            client.upsert(
                collection_name=COLLECTION_NAME,
                points=points,
            )
            return True
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                print(f"\n  Batch {batch_num} failed (attempt {attempt + 1}): {e}")
                print(f"  Retrying in {RETRY_DELAY}s...")
                time.sleep(RETRY_DELAY * (attempt + 1))  # exponential backoff
            else:
                print(f"\n  ❌ Batch {batch_num} failed after {MAX_RETRIES} attempts: {e}")
                return False
    return False

In [ ]:
# Main upload loop
uploaded = 0
failed_batches = 0
batch_num = 0

pbar = tqdm(total=total_vectors, desc="Uploading to Qdrant")
t_start = time.time()

for shard_file, meta_file in zip(shard_files, meta_files, strict=True):
    # Load shard
    embeddings = np.load(shard_file)["embeddings"]
    with open(meta_file, encoding="utf-8") as f:
        metadata = json.load(f)

    assert len(embeddings) == len(metadata), (
        f"Shard {shard_file.name}: {len(embeddings)} embeddings vs {len(metadata)} metadata"
    )

    # Upload in batches
    for i in range(0, len(embeddings), UPLOAD_BATCH_SIZE):
        batch_end = min(i + UPLOAD_BATCH_SIZE, len(embeddings))
        batch_emb = embeddings[i:batch_end]
        batch_meta = metadata[i:batch_end]

        points = []
        for j in range(len(batch_emb)):
            meta = batch_meta[j]
            point = PointStruct(
                id=make_point_id(meta["image_id"]),
                vector=batch_emb[j].tolist(),
                payload={
                    "image_id": meta["image_id"],
                    "original_class": meta["original_class"],
                    "target_class": meta.get("target_class", "unknown"),
                    "relative_path": meta["relative_path"],
                    "filename": meta["filename"],
                },
            )
            points.append(point)

        success = upload_batch_with_retry(points, batch_num)
        if success:
            uploaded += len(points)
        else:
            failed_batches += 1

        batch_num += 1
        pbar.update(len(points))

    # Free memory after each shard
    del embeddings, metadata

pbar.close()
elapsed = time.time() - t_start

print("\nUpload complete!")
print(f"  Uploaded: {uploaded} points")
print(f"  Failed batches: {failed_batches}")
print(f"  Time: {elapsed:.1f}s ({uploaded / max(elapsed, 1):.1f} points/s)")

## 8. Verification

In [ ]:
# Verify final collection state
info = client.get_collection(COLLECTION_NAME)
print(f"Collection: {COLLECTION_NAME}")
print(f"  Points: {info.points_count}")
print(f"  Vectors: {info.vectors_count}")
print(f"  Status: {info.status}")

In [ ]:
# Run a sample similarity query to verify search works
# Use the first vector from the first shard as query
sample_shard = np.load(shard_files[0])
query_vector = sample_shard["embeddings"][0].tolist()

with open(meta_files[0], encoding="utf-8") as f:
    sample_meta = json.load(f)
query_image = sample_meta[0]

print(f"Query image: {query_image['filename']} ({query_image['target_class']})")
print()

results = client.search(
    collection_name=COLLECTION_NAME,
    query_vector=query_vector,
    limit=5,
)

print("Top-5 similar images:")
for i, hit in enumerate(results, 1):
    payload = hit.payload
    print(
        f"  {i}. [{hit.score:.4f}] {payload.get('filename', 'N/A')} "
        f"({payload.get('target_class', 'N/A')})"
    )

---

## Summary

| Metric | Value |
|--------|-------|
| Collection | `fruvia_images_dinov2_base_v1` |
| Vector dimension | 768 |
| Distance metric | Cosine |
| Points uploaded | (see above) |

The Qdrant collection is now ready for the Fruvia AI backend retrieval endpoint.